# MaveDB metadata curation workflow

This notebook assembles the sealed LLM metadata export with optional human curation, processes MaveDB datasets, combines the per-dataset parquet files, then standardizes and validates the combined metadata.

Run cells in order. Update the paths in **Run inputs** first. Each run writes to a new directory; choose a new `RUN_NAME` when rerunning. BigQuery publishing is a separate, disabled-by-default step at the end.

For a non-interactive run, see `curation_tools/study_curation/mavedb/README.md`.


## Setup

In [8]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display


REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data_exploration" / "curation_tools").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the PerturbationCatalogue repository")

sys.path.insert(0, str(REPO_ROOT / "data_exploration"))

from curation_tools.study_curation.mavedb import (
    combine_dataset_parquets,
    create_run_directory,
    filter_curated_metadata,
    finalize_outputs,
    load_and_merge_metadata,
    process_mavedb_datasets,
    publish_to_bigquery,
    standardize_metadata,
)

from curation_tools.study_curation.paths import (
    CURATION_RUNS_DIR,
    DEFAULT_MAPPING_PATH,
    DEFAULT_OVERRIDE_PATH,
    MAVEDB_DIR,
    MAVEDB_DUMP_DIR,
)


## Run inputs

In [10]:
MAVEDB_ROOT = MAVEDB_DIR
RUNS_DIR = CURATION_RUNS_DIR
RUN_NAME = "my-new-run"  # choose the sealed LLM run to process
PROCESSING_NAME = "default"  # choose a fresh name for each postprocessing run
BASE_DIR = RUNS_DIR / RUN_NAME
OUTPUT_DIR = BASE_DIR / "postprocessing" / PROCESSING_NAME

# Select the CSV exported from a sealed LLM curation run.
LLM_METADATA_PATH = BASE_DIR / "exports" / "curation_final_metadata.csv"
# Historical file-based runs may instead use BASE_DIR / "step5_final" / "step5_final_metadata.csv".

# Set to None when running without the human curation spreadsheet.
MANUAL_METADATA_PATH = MAVEDB_ROOT / "mavedb_studies.xlsx"

# Controlled term mappings used during metadata standardization.
MAPPING_PATH = DEFAULT_MAPPING_PATH
OVERRIDE_PATH = DEFAULT_OVERRIDE_PATH
MAVEDB_CSV_DIR = MAVEDB_DUMP_DIR
BIGQUERY_PROJECT = "prj-ext-dev-pertcat-437314"


## 1. Assemble metadata sources

Non-empty human values take precedence. Conflicting values are returned for review; LLM-only dataset rows and columns are retained.

In [3]:
merged_metadata, source_conflicts = load_and_merge_metadata(
    llm_metadata_path=LLM_METADATA_PATH,
    manual_metadata_path=MANUAL_METADATA_PATH,
)

# Filter the merged metadata to include only curated datasets (by human or LLM; i.e., those having a "is_curated" flag) for further processing
datasets_for_processing = filter_curated_metadata(merged_metadata)
print(f"Merged datasets: {merged_metadata['dataset_id'].nunique():,}")
print(f"Curated datasets selected for processing: {datasets_for_processing['dataset_id'].nunique():,}")
print(f"Values present in both human and LLM metadata (just for inspection, as human values take precedence): {len(source_conflicts):,}")
display(source_conflicts.head(50))
print("Conflicting fields (by count):")
display(source_conflicts['field'].value_counts())


/Users/zakirov/Documents/GitHub/PerturbationCatalogue/data_exploration/curation_tools/mavedb_postprocessing/workflow.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.replace(r"^\s*$", pd.NA, regex=True)


Merged datasets: 2,677
Curated datasets selected for processing: 894
Values present in both human and LLM metadata (just for inspection, as human values take precedence): 1,527


/Users/zakirov/Documents/GitHub/PerturbationCatalogue/data_exploration/curation_tools/mavedb_postprocessing/workflow.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.replace(r"^\s*$", pd.NA, regex=True)


,dataset_id,manual_row_number,field,manual_value,llm_value
0,urn:mavedb:00000104-a-1,204,data_modality,MAVE,CRISPR screen
1,urn:mavedb:00000002-a-1,12,perturbed_target_symbol,YAP1,hYAP65 WW domain
2,urn:mavedb:00000002-a-2,13,perturbed_target_symbol,YAP1,hYAP65 WW domain
3,urn:mavedb:00000003-b-2,17,perturbed_target_symbol,BRCA1,BRCA1 RING domain
4,urn:mavedb:00000006-a-1,24,perturbed_target_symbol,ALDOB,ALDOB enhancer
5,urn:mavedb:00000007-a-1,25,perturbed_target_symbol,ECR11,ECR11 enhancer
6,urn:mavedb:00000014-a-1,28,perturbed_target_symbol,BCL11A,BCL11A enhancer
7,urn:mavedb:00000015-a-1,29,perturbed_target_symbol,F9,F9 promoter
8,urn:mavedb:00000016-a-1,30,perturbed_target_symbol,FOXE1,FOXE1 promoter
9,urn:mavedb:00000017-a-1,31,perturbed_target_symbol,GP1BB,GP1BB promoter


Conflicting fields (by count):


field
number_of_perturbed_targets         260
experiment_title                    258
experiment_summary                  237
study_title                         156
first_author                        106
last_author                          97
perturbed_target_symbol              87
study_uri                            63
species                              41
score_interpretation                 19
curation_agent_name                  19
license_label                        19
curation_agent_type                  19
readout_technology_label             14
reference_genome_label               14
associated_datasets                  14
library_generation_method_label      13
study_year                           12
software_analysis_label               9
library_total_variants                7
treatment_label                       6
treatment_dose                        6
treatment_type_label                  5
library_manufacturer                  5
readout_type_label                

## 2. Save run inputs and process datasets

The output directory must be new or empty. Per-dataset progress is shown here; detailed processing messages go to `logs/per_dataset_processing.log`.

In [ ]:
RUN_DIR = create_run_directory(OUTPUT_DIR)
for subdirectory in ("sources", "normalized", "qc"):
    (RUN_DIR / subdirectory).mkdir(parents=True, exist_ok=True)

merged_metadata.to_csv(RUN_DIR / "sources" / "merged_metadata.csv", index=False)
datasets_for_processing.to_csv(
    RUN_DIR / "sources" / "metadata_for_processing.csv", index=False
)
source_conflicts.to_csv(RUN_DIR / "qc" / "source_conflicts.csv", index=False)

processing_log = process_mavedb_datasets(
    metadata=datasets_for_processing,
    mavedb_csv_dir=MAVEDB_CSV_DIR,
    output_dir=RUN_DIR,
)
print(f"Detailed log: {processing_log}")

## 3. Combine per-dataset parquet files

In [ ]:
# Comment out when troubleshooting 

# RUN_DIR = OUTPUT_DIR

In [5]:
combined_metadata_path, combined_data_path = combine_dataset_parquets(RUN_DIR)

combined_metadata = pd.read_parquet(combined_metadata_path)
combined_data = pd.read_parquet(combined_data_path)
print(f"Combined metadata rows: {len(combined_metadata):,}")
print(f"Combined score rows: {len(combined_data):,}")


Combined metadata rows: 3,203,851
Combined score rows: 12,712,651


## 4. Standardize and review QC tables

Term mappings and study-specific corrections run on the combined metadata. Unmapped values remain unchanged and are saved for review. A dataset-level CSV without `sample_id` and `perturbation_name` is also saved as a reference when editing the mapping and override tables.

In [6]:
standardized_metadata, unmapped_terms, override_audit = standardize_metadata(
    combined_metadata,
    mapping_path=MAPPING_PATH,
    override_path=OVERRIDE_PATH,
    inplace=True,
)
standardized_metadata_path = (
    RUN_DIR / "normalized" / "mavedb_all_curated_metadata_standardized.parquet"
)
standardized_metadata.to_parquet(standardized_metadata_path, index=False)
dataset_level_metadata = (
    standardized_metadata.drop(
        columns=["sample_id", "perturbation_name"], errors="ignore"
    ).drop_duplicates()
)
dataset_level_metadata_path = (
    RUN_DIR
    / "normalized"
    / "mavedb_all_curated_metadata_standardized_per_dataset.csv"
)
dataset_level_metadata.to_csv(dataset_level_metadata_path, index=False)
print(
    f"Dataset-level metadata CSV ({len(dataset_level_metadata):,} rows): "
    f"{dataset_level_metadata_path}"
)
unmapped_terms.to_csv(RUN_DIR / "qc" / "unmapped_terms.csv", index=False)
override_audit.to_csv(RUN_DIR / "qc" / "override_audit.csv", index=False)

print(f"Unmapped term rows: {len(unmapped_terms):,}")
display(unmapped_terms.head(100))
display(override_audit.loc[override_audit["rows_affected"].gt(0)])


Dataset-level metadata CSV (894 rows): /Users/zakirov/Documents/GitHub/PerturbationCatalogue/curation_runs/rerun-28-08-2026/human_llm_merged_v2/normalized/mavedb_all_curated_metadata_standardized_per_dataset.csv
Unmapped term rows: 1


,source_field,target_field,unmapped_value,row_count,issue_type
0,treatment_label,treatment_label,sulfate-limited media,125032,unrecognized_label


,rule_id,phase,action,target_column,rows_affected,reason
0,clear_cell_line_for_other_model_systems,pre_mapping,clear,cell_line_label,2421975,Cell line is only applicable to cell-line mode...
1,clear_cell_line_for_other_model_systems,pre_mapping,clear,cell_line_id,2421975,Cell line is only applicable to cell-line mode...
2,clear_taha_abp_dose,mapping,clear,treatment_dose,6524,TAHA-ABP has no dose in this metadata record
3,clear_taha_abp_unit,mapping,clear,treatment_unit,6524,TAHA-ABP has no unit in this metadata record
4,clear_unresolved_treatment_ecr11_type_id,mapping,clear,treatment_type_id,1860,These entries were intentionally left without ...
5,clear_unresolved_treatment_ecr11_type_label,mapping,clear,treatment_type_label,1860,These entries were intentionally left without ...
6,clear_unresolved_treatment_ecr11_id,mapping,clear,treatment_id,1860,These entries were intentionally left without ...
7,clear_unresolved_treatment_ecr11_dose,mapping,clear,treatment_dose,1860,These entries were intentionally left without ...
8,clear_unresolved_treatment_ecr11_unit,mapping,clear,treatment_unit,1860,These entries were intentionally left without ...
9,clear_unresolved_treatment_gtppppytvg_type_id,mapping,clear,treatment_type_id,78670,These entries were intentionally left without ...


## 5. Validate metadata and align score data

Validation failures display a compact set of failing records and leave the pre-validation parquet available for review.

In [ ]:
# Comment out if troubleshooting 

# def replace_values(df, column, old_value, new_value):
#     mask = df[column] == old_value
#     if mask.sum() == 0:
#         print(f"No occurrences of '{old_value}' found in column '{column}'.")
#         return df
#     else:
#         df.loc[mask, column] = new_value
#         print(f"Replaced {mask.sum()} occurrences of '{old_value}' with '{new_value}' in column '{column}'.")
#         return df
    
# def map_values(df, source_column, target_column, source_value, target_value):
#     mask = df[source_column] == source_value
#     if mask.sum() == 0:
#         print(f"No occurrences of '{source_value}' found in column '{source_column}'.")
#         return df
#     else:
#         df.loc[mask, target_column] = target_value
#         print(f"Mapped {mask.sum()} occurrences of '{source_value}' from column '{source_column}' to '{target_value}' in column '{target_column}'.")
#         return df


# standardized_metadata = replace_values(standardized_metadata, 'species', 'Mus Musculus', 'Mus musculus')
# standardized_metadata = replace_values(standardized_metadata, 'species', "Streptococcus sp. 'group G'", "Streptococcus sp. group G")

# standardized_metadata = map_values(standardized_metadata, 'dataset_id', 'study_uri', 'urn:mavedb:00000010-a-1', 'https://doi.org/10.1261/rna.040709.113')
# standardized_metadata = map_values(standardized_metadata, 'dataset_id', 'study_year', 'urn:mavedb:00000010-a-1', 2013)


In [ ]:
# Comment out when troubleshooting 

# import importlib
# import curation_tools.perturbseq_anndata_schema
# importlib.reload(curation_tools.perturbseq_anndata_schema)
# from curation_tools.perturbseq_anndata_schema import ObsSchema
# from curation_tools.study_curation.mavedb.standardization import (
#     validate_with_unique_values,
# )

# validate_with_unique_values(
#     ObsSchema,
#     standardized_metadata,
#     n_failure_cases=None,
#     lazy=True
# )

In [4]:
combined_data_path = (RUN_DIR / "final" / "mavedb_all_curated_data_postfilter.parquet")
standardized_metadata_path = (RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation.parquet")

artifacts = finalize_outputs(
    standardized_metadata_path=standardized_metadata_path,
    combined_data_path=combined_data_path,
    save_joint_artifact=True,
    output_dir=RUN_DIR,
)

print(f"Validated metadata: {artifacts.metadata_path}")
print(f"Filtered score data: {artifacts.data_path}")
final_metadata = pd.read_parquet(artifacts.metadata_path)
print(f"Validated metadata rows: {len(final_metadata):,}")
display(final_metadata.head())


Validated metadata: /Users/zakirov/Documents/GitHub/PerturbationCatalogue/curation_runs/rerun-28-08-2026/human_llm_merged_v2/final/mavedb_all_curated_metadata_postvalidation.parquet
Filtered score data: /Users/zakirov/Documents/GitHub/PerturbationCatalogue/curation_runs/rerun-28-08-2026/human_llm_merged_v2/final/mavedb_all_curated_data_postfilter.parquet
Validated metadata rows: 3,203,851


,dataset_id,sample_id,cell_barcode,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,...,software_analysis_label,software_analysis_id,score_interpretation,reference_genome_label,reference_genome_id,associated_datasets,license_label,license_id,curation_agent_type,curation_agent_name
0,urn:mavedb:00000001-a-1,urn-mavedb-00000001-a-1-1,None,MAVE,None,<NA>,p.Pro73Gln,chr16:1291357-1327018;1,16,16,...,custom,None,0 = same as null control; 1 = synonymous,cDNA reference sequence,None,"[{""dataset_accession"":""urn:mavedb:00000001-a-1...",CC0 1.0,SWO:1000049,human,Aleks Zakirov
1,urn:mavedb:00000001-a-1,urn-mavedb-00000001-a-1-2,None,MAVE,None,<NA>,p.Tyr137Val,chr16:1291357-1327018;1,16,16,...,custom,None,0 = same as null control; 1 = synonymous,cDNA reference sequence,None,"[{""dataset_accession"":""urn:mavedb:00000001-a-1...",CC0 1.0,SWO:1000049,human,Aleks Zakirov
2,urn:mavedb:00000001-a-1,urn-mavedb-00000001-a-1-3,None,MAVE,None,<NA>,p.Val27Asp,chr16:1291357-1327018;1,16,16,...,custom,None,0 = same as null control; 1 = synonymous,cDNA reference sequence,None,"[{""dataset_accession"":""urn:mavedb:00000001-a-1...",CC0 1.0,SWO:1000049,human,Aleks Zakirov
3,urn:mavedb:00000001-a-1,urn-mavedb-00000001-a-1-4,None,MAVE,None,<NA>,p.Lys14Ser,chr16:1291357-1327018;1,16,16,...,custom,None,0 = same as null control; 1 = synonymous,cDNA reference sequence,None,"[{""dataset_accession"":""urn:mavedb:00000001-a-1...",CC0 1.0,SWO:1000049,human,Aleks Zakirov
4,urn:mavedb:00000001-a-1,urn-mavedb-00000001-a-1-5,None,MAVE,None,<NA>,p.Pro73Cys,chr16:1291357-1327018;1,16,16,...,custom,None,0 = same as null control; 1 = synonymous,cDNA reference sequence,None,"[{""dataset_accession"":""urn:mavedb:00000001-a-1...",CC0 1.0,SWO:1000049,human,Aleks Zakirov


## 6. Optional BigQuery publish

Review both local parquet outputs before enabling this cell. It checks both destination schemas before writing, then merges rows into the existing MaveDB tables.


#### Filter by species == Homo sapiens

In [ ]:
import polars as pl

meta_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation.parquet"))
    .filter(pl.col("species").eq("Homo sapiens"))
    .collect()
)
hs_datasets = meta_hs['dataset_id'].unique().sort()
display(meta_hs)

data_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_data_postfilter.parquet"))
    .filter(pl.col("dataset_id").is_in(hs_datasets))
    .collect()
)
display(data_hs)

meta_data_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_data_with_metadata_postfilter.parquet"))
    .filter(pl.col("species").eq("Homo sapiens"))
    .collect()
)

display(meta_data_hs)

OUTPUT_METADATA_POSTVALIDATION_HS = (RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation_homo_sapiens.parquet")
OUTPUT_DATA_POSTFILTER_HS = (RUN_DIR / "final" / "mavedb_all_curated_data_postfilter_homo_sapiens.parquet")
OUTPUT_METADATA_DATA_POSTVALIDATION_HS = (RUN_DIR / "final" / "mavedb_all_curated_metadata_data_postvalidation_homo_sapiens.parquet")

meta_hs.write_parquet(OUTPUT_METADATA_POSTVALIDATION_HS)
data_hs.write_parquet(OUTPUT_DATA_POSTFILTER_HS)
meta_data_hs.write_parquet(OUTPUT_METADATA_DATA_POSTVALIDATION_HS)

dataset_id,sample_id,perturbed_target_symbol,perturbed_target_ensg,perturbation_name,score_name,score_value,cell_barcode,data_modality,significant,significance_criteria,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint_post_transfection,differentiation_timepoint,experimental_timepoint,treatment_type_label,treatment_type_id,treatment_label,treatment_id,treatment_dose,treatment_unit,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,…,library_scope_id,library_perturbation_type_label,library_perturbation_type_id,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_label,readout_dimensionality_id,readout_type_label,readout_type_id,readout_technology_label,readout_technology_id,readout_measurement_label,readout_measurement_id,method_name_label,method_name_id,method_uri,sequencing_library_kit_label,sequencing_library_kit_id,sequencing_platform_label,sequencing_platform_id,sequencing_strategy_label,sequencing_strategy_id,software_counts_label,software_counts_id,software_analysis_label,software_analysis_id,score_interpretation,reference_genome_label,reference_genome_id,associated_datasets,license_label,license_id,curation_agent_type,curation_agent_name
str,str,str,str,str,str,f64,null,str,null,str,str,str,i64,i64,str,null,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,null,str,str,str,str,str,str,i64,str,null,str,str,str,str,str,str,str,str,str,str,null,str,str,str,str,str,null,str,null,str,str,null,str,str,str,str,str
"""urn:mavedb:00000001-a-1""","""urn-mavedb-00000001-a-1-1""","""UBE2I""","""ENSG00000103275""","""p.Pro73Gln""","""score""",-0.026176,null,"""MAVE""",null,null,"""chr16:1291357-1327018;1""","""16""",16,1,"""protein_coding""",null,"""DMS""",null,"""P2D""",null,"""P2D""",null,null,null,null,null,null,null,null,"""yeast""","""NCIT:C19617""","""Homo sapiens""",null,null,null,null,…,null,"""mutagenesis""","""NCIT:C17376""","""Roth lab""",null,null,null,6553,"""single-dimensional assay""",null,"""phenotypic""","""EFO:0920062""","""population growth assay""","""EFO:0002907""","""cell proliferation""","""BAO:0002805""","""Joined and refined DMS-BarSeq …",null,"""https://bitbucket.org/rothlabt…","""Nextera XT DNA Library Prepara…",null,"""Illumina NextSeq 500""","""EFO:0009173""","""barcode sequencing|direct sequ…",null,"""custom""",null,"""custom""",null,"""0 = same as null control; 1 = …","""cDNA reference sequence""",null,"""[{""dataset_accession"":""urn:mav…","""CC0 1.0""","""SWO:1000049""","""human""","""Aleks Zakirov"""
"""urn:mavedb:00000001-a-1""","""urn-mavedb-00000001-a-1-2""","""UBE2I""","""ENSG00000103275""","""p.Tyr137Val""","""score""",-0.115652,null,"""MAVE""",null,null,"""chr16:1291357-1327018;1""","""16""",16,1,"""protein_coding""",null,"""DMS""",null,"""P2D""",null,"""P2D""",null,null,null,null,null,null,null,null,"""yeast""","""NCIT:C19617""","""Homo sapiens""",null,null,null,null,…,null,"""mutagenesis""","""NCIT:C17376""","""Roth lab""",null,null,null,6553,"""single-dimensional assay""",null,"""phenotypic""","""EFO:0920062""","""population growth assay""","""EFO:0002907""","""cell proliferation""","""BAO:0002805""","""Joined and refined DMS-BarSeq …",null,"""https://bitbucket.org/rothlabt…","""Nextera XT DNA Library Prepara…",null,"""Illumina NextSeq 500""","""EFO:0009173""","""barcode sequencing|direct sequ…",null,"""custom""",null,"""custom""",null,"""0 = same as null control; 1 = …","""cDNA reference sequence""",null,"""[{""dataset_accession"":""urn:mav…","""CC0 1.0""","""SWO:1000049""","""human""","""Aleks Zakirov"""
"""urn:mavedb:00000001-a-1""","""urn-mavedb-00000001-a-1-3""","""UBE2I""","""ENSG00000103275""","""p.Val27Asp""","""score""",-0.06

In [ ]:
# change to True when you're confident the data is ready to be published to BigQuery
PUBLISH_TO_BIGQUERY = False

if PUBLISH_TO_BIGQUERY:
    publish_to_bigquery(
        metadata_path=OUTPUT_METADATA_POSTVALIDATION_HS,
        data_path=OUTPUT_DATA_POSTFILTER_HS,
        project_id=BIGQUERY_PROJECT,
    )

python(12970) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Using provided project_id: prj-ext-dev-pertcat-437314


python(12987) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Staging table: loading `.parquet` file /Users/zakirov/Documents/GitHub/PerturbationCatalogue/curation_runs/rerun-28-08-2026/human_llm_merged_v2/final/mavedb_all_curated_metadata_postvalidation_homo_sapiens.parquet to prj-ext-dev-pertcat-437314.mavedb.metadata_staging...
Staging table: loaded 1704850 rows to prj-ext-dev-pertcat-437314.mavedb.metadata_staging


python(13038) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.mavedb.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.mavedb.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.mavedb.metadata_staging
Using provided project_id: prj-ext-dev-pertcat-437314


python(13640) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Staging table: loading `.parquet` file /Users/zakirov/Documents/GitHub/PerturbationCatalogue/curation_runs/rerun-28-08-2026/human_llm_merged_v2/final/mavedb_all_curated_data_postfilter_homo_sapiens.parquet to prj-ext-dev-pertcat-437314.mavedb.data_staging...
Staging table: loaded 8171025 rows to prj-ext-dev-pertcat-437314.mavedb.data_staging


python(13681) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.mavedb.data_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.mavedb.data with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.mavedb.data_staging
